# Day 9 — ILT 1: Need for Incremental Loading — Strategies and Trade-offs

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Follows** | Day 5-6 (Silver/Gold full loads), Day 2-4 (ingestion patterns) |
| **Duration** | 60 minutes |
| **Format** | Instructor-led — run every cell live against the real `gbmart` catalog (GlobalMart's actual run; your own catalog will be named differently) |

### Learning Objectives
- Explain why re-scanning entire tables on every run stops being viable as data grows
- Name the three incremental-loading strategies already present in GlobalMart's real pipeline, and what each one tracks
- Compare their trade-offs: what each strategy catches, and what it silently misses

### Prerequisites
- Day 2 (Lakeflow Connect CDC pipeline), Day 3-4 (Autoloader), Day 5 (Silver build)

---

## The Problem

Every notebook you've written so far in this course does a **full load**: read the entire source, transform all of it, overwrite or fully rebuild the target. That's fine when `products` is 500 rows. It stops being fine when `orders` is 126,000 rows and growing by a few hundred a day — re-processing all 126,000 rows just to pick up 200 new ones wastes compute, wastes time, and at real production scale (millions of rows) can turn a 2-minute job into a 2-hour job.

**Simple way to think about it:** imagine re-reading an entire 500-page book from page 1 every single morning, just to find the one new paragraph someone added overnight. That's what a full load does. Incremental loading is the equivalent of a bookmark — you jump straight to where you left off.

**Aviskar M Dhavale** ran into exactly this last week: a Bronze full-reload job that used to finish in under 2 minutes was suddenly taking 40 minutes. Nothing about the code had changed — the source table had just kept growing, run after run, while the job kept re-reading all of it from scratch every time. That's the failure mode this whole session is about.

**Incremental loading** means: on every run, touch only the rows that are new or changed since the last run. The hard part is knowing *which rows those are* — and it turns out there's more than one way to answer that question. GlobalMart's real pipeline already uses **three different answers**, for three different reasons. This ILT walks through all three before Day 9's hands-on labs build two of them yourself.

## The Three Strategies Already Running in GlobalMart's Pipeline

| # | Strategy | What it tracks | Where GlobalMart uses it | Taught in |
|---|---|---|---|---|
| 1 | **File checkpoint** (Autoloader) | *Which files* have already been read | `customers`, `products`, `addresses`, `payments`, `payment_methods`, `returns` — all ADLS file drops | Day 3-4 |
| 2 | **Cursor / watermark column** | *Which rows* have a business timestamp newer than last time | `orders`, `order_items` — via the real Lakeflow Connect pipeline `orders_data_ingestion_cdc` | Day 2 (concept), Day 9 (build it yourself) |
| 3 | **Delta Change Data Feed (CDF)** | *Which rows* changed, at the Delta-transaction level, including deletes | Bronze → Silver refresh, once CDF is enabled | Day 9 |

These are genuinely different mechanisms solving genuinely different problems — not three names for the same idea. The rest of this ILT is about *why* GlobalMart needed three, not just one.

In [ ]:
# Live, read-only — shows the real scale that makes "just re-read everything" expensive.
# This is the same gbmart.bronze.orders / order_items you built the Lakeflow Connect
# pipeline for back in Day 2 — the row counts here are why incremental loading matters.
orders_count = spark.sql("SELECT COUNT(*) AS c FROM gbmart.bronze.orders").collect()[0]["c"]
order_items_count = spark.sql("SELECT COUNT(*) AS c FROM gbmart.bronze.order_items").collect()[0]["c"]

print(f"gbmart.bronze.orders       : {orders_count:,} rows")
print(f"gbmart.bronze.order_items  : {order_items_count:,} rows")
print()
print("A full re-scan of order_items to find ~200 new rows means reading all of the above,")
print("every single run, forever. That cost only grows as GlobalMart grows.")

## Strategy 1 (recap) — File Checkpoint (Autoloader)

Already fully covered in Day 3-4: Autoloader's `cloudFiles.schemaLocation` + `checkpointLocation` remember which **files** have already been streamed into Bronze. Drop a new file in, re-run, only the new file gets processed.

**Simple way to think about it:** it's like a checklist of filenames you've already opened. Autoloader only ever asks "is this filename new to me?" — never "did the contents of a filename I already checked off actually change?"

**What it's good at:** cheap, simple, works great when each new batch of data arrives as a brand-new file (a new day's `customers_020626.csv`, a new `products_020626.json`).

**What it silently misses:** a change made *inside a file Autoloader has already processed*. **Bysani Karthik** caught this while reviewing the Bronze `payment_methods` job: someone had hand-corrected a typo inside an already-ingested `payment_methods` file and re-uploaded it under the exact same filename, expecting Bronze to pick up the fix on the next run. It didn't — as far as Autoloader's checkpoint is concerned, that filename is already done. File-level tracking has no idea what happened *inside* a file it already saw.

## Strategy 2 — Cursor / Watermark Column

This is what the real `orders_data_ingestion_cdc` pipeline does (Day 2). Instead of tracking files, it tracks a **business column** — `updated_at` — and on every run asks the source: *"give me every row where `updated_at` is newer than the last value I saw."* The pipeline remembers that last value (its "watermark" or "cursor") and advances it after each successful run.

**Simple way to think about it:** it's like asking a librarian "show me every book returned after 3pm yesterday" instead of re-checking every book in the library. It only works because the library reliably stamps a return time on every book — in GlobalMart's case, that reliable stamp is a real Postgres trigger, `trg_orders_updated_at`, which auto-bumps `updated_at` on every single update to `orders`. No trigger, no trustworthy watermark.

```
table_configuration: {
  primary_keys: ["orderid"],
  query_based_connector_config: { cursor_columns: ["updated_at"] }
}
```

**What it's good at:** works on any table with a reliable, always-bumped timestamp column — doesn't require the source to support anything special (no logical replication, no CDF, just a `WHERE updated_at > X` query). This is exactly why it's the right choice for reading a Postgres OLTP table over a plain connection.

**What it silently misses — and this is the trade-off worth remembering from Day 2:** a **hard DELETE**. If a row is deleted from the source table, there's no row left with a newer `updated_at` to signal that — the deleted row just vanishes from the next query result, and the cursor-based pipeline has no event to tell it "this row is gone." **Amardeep meena** flagged exactly this gap while walking through the Day 2 pipeline design with the finance team: if an order is ever hard-deleted in Postgres, `orders_data_ingestion_cdc` will not remove it from Bronze — it will just quietly stop appearing in future *changes*, while the old row stays behind. GlobalMart's real `orders_data_ingestion_cdc` pipeline has exactly this blind spot today.

## Strategy 3 — Delta Change Data Feed (CDF)

CDF is different from both strategies above: it doesn't watch files, and it doesn't rely on a business column at all. It's a **Delta Lake feature** — once turned on for a table, every `INSERT` / `UPDATE` / `DELETE` against that table is automatically recorded as a row-level change event, tagged with `_change_type`, `_commit_version`, and `_commit_timestamp`. A downstream reader asks Delta "give me everything that changed since version N" and gets back exactly that — including deletes, which is the one thing Strategy 2 can't see.

**Simple way to think about it:** think of it as a security camera pointed at the table instead of a snapshot photo of it. A snapshot only shows you what the table looks like *right now* — a camera shows you every single thing that happened to it, in order, including things that got deleted and are no longer visible in any snapshot.

**Aman Shankar Verma** put it well when explaining this to the cohort: "CDF is the only one of the three that would actually tell you a row got deleted — not just that it's missing."

**What it's good at:** the most complete picture — inserts, updates, *and* deletes, all captured automatically with zero reliance on the source having a well-behaved timestamp column.

**What it costs:** it's Delta-to-Delta only — it's not something you can point at an external Postgres source. In GlobalMart's real pipeline, CDF is enabled today on **Bronze**, and Silver reads it to refresh incrementally from Bronze. CDF is *also* enabled on **Silver**, but nothing consumes it yet: Gold (`dim_*` / `fact_sales`) is still built by full overwrite, not incremental `MERGE`. Enabling CDF on Silver now is about staging it for Day 12's incremental SCD work — it's ready, it's just not being read yet. It also has a retention/storage cost (`delta.enableChangeDataFeed` keeps a changelog you must eventually vacuum with a retention window in mind, same as `VACUUM` and time-travel history).

## Side-by-Side Trade-offs

| | File Checkpoint | Cursor / Watermark | Delta CDF |
|---|---|---|---|
| Tracks | Files already read | Rows newer than a timestamp | Row-level Delta transactions |
| Catches inserts | Yes (new file) | Yes | Yes |
| Catches updates | Only if in a new file | Yes, if `updated_at` bumped | Yes, always |
| Catches hard deletes | No | **No** | **Yes** |
| Works against | File drops (ADLS) | Any queryable source (e.g. Postgres) | Delta tables only |
| Setup cost | Low (built into Autoloader) | Low-medium (needs a trustworthy timestamp column + trigger) | Low (one `ALTER TABLE`), but Delta-only |
| Used in GlobalMart for | 6 of 8 Bronze sources | `orders` / `order_items` ingestion | Bronze → Silver refresh |

**The takeaway:** there's no single "best" strategy — each one is the right tool for a specific kind of source and a specific kind of gap you're willing to accept. Day 9's two hands-on labs build the watermark strategy yourself (so the Day 2 pipeline stops being a black box) and wire up real CDF-based Bronze→Silver incremental loading.

## Instructor Discussion Questions
1. GlobalMart's finance team asks: "can we trust `fact_sales` if a customer cancels and deletes their account?" Which of the three strategies would silently miss that, and where in the pipeline would you need to check?
2. Why might GlobalMart choose the cursor/watermark strategy for `orders` from Postgres, rather than enabling something CDF-like *inside* Postgres itself?
3. If `payment_methods` (currently Autoloader/file-checkpoint) started receiving in-place row edits instead of new files, what would break, and which strategy would fix it?